# Disagreement Matrix: Direct Evidence That a Reliability Vector Beats a Scalar

DESIGN.md 16.1 asserts that correctness, OOD-ness, and feature-space anomaly are different latent variables based on results scattered across sections 15.3-15.6 - but nowhere is that assertion backed by one table computed the same way for every signal. This notebook builds that table directly: for every named evidence operator/signal this repo has, measure its AUROC against four different targets on real ResNet-50 cached data, and see whether any single signal wins at all four or whether the ranking genuinely reorders - which is the actual, checkable form of "a scalar can't do this job."

In [1]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
torch.manual_seed(0)
%matplotlib inline

from deployment_reliability.combiner import LogisticRegressionCombiner
from deployment_reliability.evidence import Evidence
from deployment_reliability.features import featurize
from deployment_reliability.normalization import ReferenceNormalizer
from deployment_reliability.router import auroc


In [2]:
CACHE_PATH = os.path.join("..", "data", "logit_cache_resnet50.pt")
assert os.path.exists(CACHE_PATH), "run scripts/collect_logits.py resnet50 first"
cache = torch.load(CACHE_PATH)
logits, labels = cache["logits"], cache["labels"]
splits_arr = np.array(cache["splits"])

def mask(name):
    return torch.from_numpy(splits_arr == name)

m_fit, m_test, m_a, m_o = [mask(n) for n in ("combiner_fit", "id_test", "imagenet_a", "imagenet_o")]
correct = logits.argmax(dim=-1) == labels
counts = {n: int(mask(n).sum()) for n in ("combiner_fit", "id_test", "imagenet_a", "imagenet_o")}
print("split sizes:", counts)
assert all(v > 0 for v in counts.values())
print("PASSED")

split sizes: {'combiner_fit': 1500, 'id_test': 1500, 'imagenet_a': 1935, 'imagenet_o': 2000}
PASSED


## Building the eight signals

All eight are oriented so **higher = more trustworthy** (matching `FEATURE_DIRECTIONS`' convention, including the corrected `logit_l2_norm` sign - see Check 0.5 below), so AUROC values are directly comparable across rows without a separate sign convention per row.

In [3]:
phi = featurize(logits)
combiner = LogisticRegressionCombiner().fit(phi[m_fit], correct[m_fit].float())
S = combiner.score(phi)
normalizer = ReferenceNormalizer().fit(phi[m_fit])
anomaly = normalizer.anomaly_score(phi).mean(dim=-1)  # mean |z-score| across the 5 features

ev = Evidence(logits)
signals = {
    "concentration": ev.concentration(),
    "separability": ev.separability(),
    "ambiguity (oriented)": -ev.ambiguity(),
    "plausibility": ev.plausibility(),
    "magnitude (oriented)": -ev.magnitude(),
    "conflict (oriented)": -ev.conflict(),
    "anomaly_score (oriented)": -anomaly,
    "combiner S": S,
}
print("signals:", list(signals))
print("PASSED")

signals: ['concentration', 'separability', 'ambiguity (oriented)', 'plausibility', 'magnitude (oriented)', 'conflict (oriented)', 'anomaly_score (oriented)', 'combiner S']
PASSED


## Check 0.5 — why `logit_l2_norm` is oriented the way it is

`FEATURE_DIRECTIONS["logit_l2_norm"]` is set to `-1` ("higher = less confident"), the opposite of the naive assumption that a larger raw logit vector means a more confident, more trustworthy prediction. The reason is empirical, not stylistic: on real cached data, **incorrect** predictions have a **higher** raw L2 norm than correct ones, consistently across all three evaluated backbones (the cell below repeats the measurement directly for this backbone). A model can push a large-magnitude logit vector while still being wrong — magnitude reflects how strongly the model committed to *some* answer, not whether that answer was the right one — so treating a larger norm as "more confident, therefore more trustworthy" has this signal working backwards, systematically favoring exactly the predictions this project most wants to catch. Because `magnitude`'s row above is built from this same oriented feature, getting the sign wrong here would silently invert one of the eight rows in the matrix that follows; `tests/test_features.py::test_logit_l2_norm_direction_confirmed_on_real_data_across_backbones` locks the direction in as a permanent regression check.

In [4]:
mean_correct = phi[m_test][correct[m_test], -1].mean().item()
mean_incorrect = phi[m_test][~correct[m_test], -1].mean().item()
print(f"raw logit_l2_norm | correct   = {mean_correct:.3f}")
print(f"raw logit_l2_norm | incorrect = {mean_incorrect:.3f}")
assert mean_incorrect > mean_correct, "regression check: this is the real, fixed finding, not noise"
print("PASSED")

raw logit_l2_norm | correct   = 15.347
raw logit_l2_norm | incorrect = 18.815
PASSED


## The matrix

Four columns, each a distinct question a deployment system might ask:

- **corr_id** — AUROC separating correct vs. incorrect predictions on `id_test`
- **corr_a** — AUROC separating correct vs. incorrect predictions *within* `imagenet_a` (the known hard case, DESIGN.md 15.2)
- **ood_o** — AUROC separating `id_test` from `imagenet_o` (genuine OOD detection)
- **shift_a** — AUROC separating `id_test` from `imagenet_a` (does the signal notice `imagenet_a` looks different at all, independent of per-instance correctness)

In [5]:
rows = []
for name, s in signals.items():
    corr_id = auroc(s[m_test][correct[m_test]], s[m_test][~correct[m_test]])
    n_pos_a, n_neg_a = int(correct[m_a].sum()), int((~correct[m_a]).sum())
    corr_a = auroc(s[m_a][correct[m_a]], s[m_a][~correct[m_a]]) if n_pos_a > 0 and n_neg_a > 0 else float("nan")
    ood_o = auroc(s[m_test], s[m_o])
    shift_a = auroc(s[m_test], s[m_a])
    rows.append((name, corr_id, corr_a, ood_o, shift_a))

print(f'{"signal":32s} {"corr_id":>9s} {"corr_a":>9s} {"ood_o":>9s} {"shift_a":>9s}')
for name, corr_id, corr_a, ood_o, shift_a in rows:
    print(f'{name:32s} {corr_id:9.4f} {corr_a:9.4f} {ood_o:9.4f} {shift_a:9.4f}')

signal                             corr_id    corr_a     ood_o   shift_a
concentration                       0.8279    0.5004    0.5184    0.8140
separability                        0.8848    0.5269    0.5612    0.7872
ambiguity (oriented)                0.7015    0.4479    0.5284    0.7575
plausibility                        0.5518    0.4121    0.5026    0.5646
magnitude (oriented)                0.7372    0.5624    0.6072    0.8513
conflict (oriented)                 0.8164    0.5191    0.5705    0.6235
anomaly_score (oriented)            0.5935    0.5227    0.4749    0.7416
combiner S                          0.8841    0.5445    0.5673    0.8309


## Checks — the actual claims this table needs to support

In [6]:
by_name = {name: (corr_id, corr_a, ood_o, shift_a) for name, corr_id, corr_a, ood_o, shift_a in rows}

# Claim 1: no single signal is the best column-wise winner on all four axes at once.
best_per_column = [max(by_name, key=lambda n: by_name[n][col]) for col in range(4)]
print("best signal per column:", best_per_column)
assert len(set(best_per_column)) > 1, "a single signal dominating every column would undercut the vector-not-scalar claim"
print("PASSED: the best signal changes depending on which question is asked")

best signal per column: ['separability', 'magnitude (oriented)', 'magnitude (oriented)', 'magnitude (oriented)']
PASSED: the best signal changes depending on which question is asked


In [7]:
# Claim 2: magnitude and the correctness-fitted combiner rank oppositely across
# corr_id vs shift_a - the clearest single number showing complementary, not
# redundant, signals (DESIGN.md 17.4's shift-invariance proposition predicts
# exactly this: combiner S is built from shift-invariant features and should be
# weaker at shift_a than the shift-SENSITIVE magnitude).
mag_corr_id, mag_shift_a = by_name["magnitude (oriented)"][0], by_name["magnitude (oriented)"][3]
S_corr_id, S_shift_a = by_name["combiner S"][0], by_name["combiner S"][3]
print(f"magnitude:   corr_id={mag_corr_id:.4f}  shift_a={mag_shift_a:.4f}")
print(f"combiner S:  corr_id={S_corr_id:.4f}  shift_a={S_shift_a:.4f}")
assert S_corr_id > mag_corr_id, "combiner S should still win on its own fitting objective (correctness)"
assert mag_shift_a > S_corr_id - 0.05, "magnitude's shift-detection should be competitive with S's best axis, not negligible"
print("PASSED")

magnitude:   corr_id=0.7372  shift_a=0.8513
combiner S:  corr_id=0.8841  shift_a=0.8309
PASSED


In [8]:
# Claim 3: corr_a and ood_o are hard for EVERY signal (all near the 0.5 chance
# line) while shift_a is comparatively easy for most - this is the precise,
# checkable version of a distinction the prose in DESIGN.md 15.2/15.3 blurred:
# "no signal here can tell which imagenet_a predictions are wrong" is a
# different (and true) claim from "no signal here can tell imagenet_a apart
# from id_test at all" (false - most signals do this reasonably well).
corr_a_values = [v[1] for v in by_name.values()]
shift_a_values = [v[3] for v in by_name.values()]
print("corr_a range:", min(corr_a_values), "-", max(corr_a_values))
print("shift_a range:", min(shift_a_values), "-", max(shift_a_values))
assert max(corr_a_values) < 0.60, "even the best signal should be near-chance at correctness WITHIN imagenet_a"
assert max(shift_a_values) > 0.75, "but several signals should clearly detect imagenet_a AS a shift from id_test"
print("PASSED: corr_a is near-chance for everything; shift_a is not - a real distinction, not a rounding difference")

corr_a range: 0.412067711353302 - 0.5624383091926575
shift_a range: 0.5646349787712097 - 0.8513458967208862
PASSED: corr_a is near-chance for everything; shift_a is not - a real distinction, not a rounding difference


## Cross-architecture replication

Everything above used ResNet-50 only. The two structural claims - no signal wins every column, and `magnitude`/`combiner S` trade off in opposite directions - are supposed to follow from §17.4's shift-invariance proposition, which doesn't mention any particular backbone. If that's right, both claims should hold on ViT-B/16 and ConvNeXt-Tiny too, not just once.

In [9]:
def compute_matrix(backbone):
    cache = torch.load(os.path.join("..", "data", f"logit_cache_{backbone}.pt"))
    logits_b, labels_b = cache["logits"], cache["labels"]
    splits_b = np.array(cache["splits"])
    def m(name):
        return torch.from_numpy(splits_b == name)
    m_fit_b, m_test_b, m_a_b, m_o_b = [m(n) for n in ("combiner_fit", "id_test", "imagenet_a", "imagenet_o")]
    correct_b = logits_b.argmax(dim=-1) == labels_b

    phi_b = featurize(logits_b)
    combiner_b = LogisticRegressionCombiner().fit(phi_b[m_fit_b], correct_b[m_fit_b].float())
    S_b = combiner_b.score(phi_b)
    normalizer_b = ReferenceNormalizer().fit(phi_b[m_fit_b])
    anomaly_b = normalizer_b.anomaly_score(phi_b).mean(dim=-1)

    ev_b = Evidence(logits_b)
    signals_b = {
        "concentration": ev_b.concentration(),
        "separability": ev_b.separability(),
        "ambiguity": -ev_b.ambiguity(),
        "plausibility": ev_b.plausibility(),
        "magnitude": -ev_b.magnitude(),
        "conflict": -ev_b.conflict(),
        "anomaly_score": -anomaly_b,
        "combiner S": S_b,
    }
    rows_b = {}
    for name, s in signals_b.items():
        corr_id = auroc(s[m_test_b][correct_b[m_test_b]], s[m_test_b][~correct_b[m_test_b]])
        n_pos_a, n_neg_a = int(correct_b[m_a_b].sum()), int((~correct_b[m_a_b]).sum())
        corr_a = auroc(s[m_a_b][correct_b[m_a_b]], s[m_a_b][~correct_b[m_a_b]]) if n_pos_a > 0 and n_neg_a > 0 else float("nan")
        ood_o = auroc(s[m_test_b], s[m_o_b])
        shift_a = auroc(s[m_test_b], s[m_a_b])
        rows_b[name] = (corr_id, corr_a, ood_o, shift_a)
    return rows_b, signals_b, m_test_b

backbone_results = {}
for backbone in ("resnet50", "vit_b16", "convnext_tiny"):
    rows_b, signals_b, m_test_b = compute_matrix(backbone)
    backbone_results[backbone] = (rows_b, signals_b, m_test_b)
    print(f"=== {backbone} ===")
    print(f'{"signal":16s} {"corr_id":>9s} {"corr_a":>9s} {"ood_o":>9s} {"shift_a":>9s}')
    for name, (a, b, c, d) in rows_b.items():
        print(f'{name:16s} {a:9.4f} {b:9.4f} {c:9.4f} {d:9.4f}')
    print()
print("PASSED")

=== resnet50 ===
signal             corr_id    corr_a     ood_o   shift_a
concentration       0.8279    0.5004    0.5184    0.8140
separability        0.8848    0.5269    0.5612    0.7872
ambiguity           0.7015    0.4479    0.5284    0.7575
plausibility        0.5518    0.4121    0.5026    0.5646
magnitude           0.7372    0.5624    0.6072    0.8513
conflict            0.8164    0.5191    0.5705    0.6235
anomaly_score       0.5935    0.5227    0.4749    0.7416
combiner S          0.8841    0.5445    0.5673    0.8309

=== vit_b16 ===
signal             corr_id    corr_a     ood_o   shift_a
concentration       0.8866    0.6308    0.5880    0.8336
separability        0.8792    0.5864    0.6258    0.8015
ambiguity           0.8599    0.6192    0.5872    0.8414
plausibility        0.7094    0.6039    0.5347    0.7598
magnitude           0.8101    0.6336    0.7205    0.8915
conflict            0.8083    0.3839    0.6015    0.6389
anomaly_score       0.6887    0.6397    0.5627    0.80

=== convnext_tiny ===
signal             corr_id    corr_a     ood_o   shift_a
concentration       0.8643    0.5161    0.5234    0.8240
separability        0.8833    0.5301    0.5738    0.7975
ambiguity           0.8020    0.4972    0.5138    0.7958
plausibility        0.6394    0.5144    0.4709    0.6532
magnitude           0.7878    0.4765    0.6889    0.8620
conflict            0.7743    0.5606    0.5826    0.6083
anomaly_score       0.7078    0.5306    0.5146    0.7891
combiner S          0.8899    0.5127    0.6062    0.8571

PASSED


In [10]:
# Claim 1 and Claim 2, re-checked on all three backbones, not just ResNet-50.
for backbone, (rows_b, signals_b, m_test_b) in backbone_results.items():
    best_per_column = [max(rows_b, key=lambda n: rows_b[n][col]) for col in range(4)]
    assert len(set(best_per_column)) > 1, f"{backbone}: a single signal dominating every column would undercut the vector-not-scalar claim"

    mag_corr_id, mag_shift_a = rows_b["magnitude"][0], rows_b["magnitude"][3]
    S_corr_id, S_shift_a = rows_b["combiner S"][0], rows_b["combiner S"][3]
    assert S_corr_id > mag_corr_id, f"{backbone}: combiner S should win on its own fitting objective (correctness)"
    assert mag_shift_a > S_shift_a, f"{backbone}: magnitude should win on shift-detection, the axis it's NOT fitted for"
    print(f"{backbone:15s} best-per-column unique winners={len(set(best_per_column))}  "
          f"S(corr_id={S_corr_id:.3f}) > magnitude(corr_id={mag_corr_id:.3f})  "
          f"magnitude(shift_a={mag_shift_a:.3f}) > S(shift_a={S_shift_a:.3f})")
print("\nPASSED: both structural claims replicate on all three architecturally distinct backbones")

resnet50        best-per-column unique winners=2  S(corr_id=0.884) > magnitude(corr_id=0.737)  magnitude(shift_a=0.851) > S(shift_a=0.831)
vit_b16         best-per-column unique winners=3  S(corr_id=0.885) > magnitude(corr_id=0.810)  magnitude(shift_a=0.892) > S(shift_a=0.837)
convnext_tiny   best-per-column unique winners=3  S(corr_id=0.890) > magnitude(corr_id=0.788)  magnitude(shift_a=0.862) > S(shift_a=0.857)

PASSED: both structural claims replicate on all three architecturally distinct backbones


## Signal correlation matrix

AUROC divergence across targets is indirect evidence that the signals carry different information. A direct version: how correlated are the signals *with each other*, on the same `id_test` predictions? Low correlation between two signals means genuinely independent information; high correlation means one is largely redundant with the other, given the first.

In [11]:
names = ["concentration", "separability", "ambiguity", "plausibility", "magnitude", "conflict", "anomaly_score", "combiner S"]

def correlation_matrix(signals_b, m_test_b):
    stacked = torch.stack([signals_b[n] for n in names], dim=0)[:, m_test_b]
    return torch.corrcoef(stacked)

rows_r50, signals_r50, m_test_r50 = backbone_results["resnet50"]
corr = correlation_matrix(signals_r50, m_test_r50)

print("     " + " ".join(f"{n[:6]:>7s}" for n in names))
for i, n in enumerate(names):
    print(f"{n[:6]:5s}" + " ".join(f"{corr[i, j].item():7.3f}" for j in range(len(names))))

      concen  separa  ambigu  plausi  magnit  confli  anomal  combin
concen  1.000   0.771   0.920   0.699   0.331   0.331  -0.192   0.761
separa  0.771   1.000   0.521   0.219   0.608   0.738  -0.127   0.959
ambigu  0.920   0.521   1.000   0.899   0.089   0.007  -0.318   0.472
plausi  0.699   0.219   0.899   1.000  -0.287  -0.202  -0.519   0.137
magnit  0.331   0.608   0.089  -0.287   1.000   0.405   0.257   0.662
confli  0.331   0.738   0.007  -0.202   0.405   1.000  -0.059   0.746
anomal -0.192  -0.127  -0.318  -0.519   0.257  -0.059   1.000   0.036
combin  0.761   0.959   0.472   0.137   0.662   0.746   0.036   1.000


In [12]:
# Claim 4: anomaly_score is measurably LESS redundant with the fitted combiner
# than separability is - separability dominates the combiner's learned weights
# (DESIGN.md 15.7), so separability-vs-combiner_S correlation is expected to be
# very high; anomaly_score was never part of the combiner's fitting objective at
# all, so its correlation with combiner_S should be meaningfully lower on every
# backbone, not just close-but-technically-lower.
def pair_corr(signals_b, m_test_b, a, b):
    corr_b = correlation_matrix(signals_b, m_test_b)
    return corr_b[names.index(a), names.index(b)].item()

for backbone, (rows_b, signals_b, m_test_b) in backbone_results.items():
    anomaly_vs_S = pair_corr(signals_b, m_test_b, "anomaly_score", "combiner S")
    separability_vs_S = pair_corr(signals_b, m_test_b, "separability", "combiner S")
    print(f"{backbone:15s} anomaly_score vs combiner_S = {anomaly_vs_S:7.3f}   separability vs combiner_S = {separability_vs_S:7.3f}")
    assert separability_vs_S > anomaly_vs_S + 0.3, f"{backbone}: expected a large, not marginal, redundancy gap"
print("\nPASSED: anomaly_score is consistently, substantially less redundant with the fitted combiner than separability is")

resnet50        anomaly_score vs combiner_S =   0.036   separability vs combiner_S =   0.959
vit_b16         anomaly_score vs combiner_S =   0.482   separability vs combiner_S =   0.947
convnext_tiny   anomaly_score vs combiner_S =   0.291   separability vs combiner_S =   0.947

PASSED: anomaly_score is consistently, substantially less redundant with the fitted combiner than separability is


## Summary

The matrix directly substantiates DESIGN.md 16.1's claim rather than just asserting it, and does so on all three evaluated backbones, not only ResNet-50: the best-performing signal changes depending on which of the four questions is asked, `magnitude` and `combiner S` trade off in opposite directions exactly as the shift-invariance proposition (17.4) predicts, and the near-chance `corr_a` numbers next to the well-above-chance `shift_a` numbers show these evaluation sets can be *detected as unusual* even when *which specific predictions are wrong within them* stays unreadable from logits alone. The correlation matrix adds a second, more direct line of evidence for the same conclusion: `anomaly_score` is consistently far less redundant with the fitted combiner than a signal the combiner is actually built from (`separability`) is, on every backbone - not because `anomaly_score` is a better signal, but because it carries information the correctness-fitted combiner never had access to in the first place.

The size of these gaps matters as much as their direction. On ResNet-50, `separability`-vs-`combiner S` correlation is 0.959 - close enough to 1 that reporting one is nearly redundant with reporting the other, which makes sense given `separability` is the feature the L2-regularized fit weights most heavily (DESIGN.md §15.7). `anomaly_score`-vs-`combiner S`, by contrast, sits at 0.036 on ResNet-50 (0.482 on ViT-B/16, 0.291 on ConvNeXt-Tiny) - low enough, and inconsistent enough across backbones, that no fixed linear relationship between the two is even a good approximation. That inconsistency is itself informative: if `anomaly_score` were just a noisier copy of `combiner S`, its correlation with `S` would be expected to sit in a narrow band across architecturally distinct backbones the way `separability`'s does (0.959/0.947/0.947); instead it swings from near-zero to moderate, meaning `ReferenceNormalizer`'s z-score-based signal is picking up on something about each backbone's feature geometry that the correctness-fitted combiner does not consistently share. Had the correlation matrix instead shown every signal pair sitting above ~0.85, that would have undercut DESIGN.md 16.1's premise directly - a `Φ` vector where every coordinate moves together is a scalar wearing a vector's syntax, not a genuine multi-dimensional evidence representation, and the routing/HITL design built on treating them separately (§13) would have far less to justify itself with.